# NB08: Synthesis & Summary Statistics

Aggregate results from NB03-NB07 into summary tables and figures
for the final report.

**Requires**: All prior notebook outputs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path('../data')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 10})

## 1. Cohort Summary Table

In [ ]:
cult_meta = pd.read_csv(DATA_DIR / 'cultured_genome_metadata.tsv', sep='\t')
cult_ko = pd.read_csv(DATA_DIR / 'cultured_ko_profiles.tsv', sep='\t')

mag_meta_path = DATA_DIR / 'mag_metadata.tsv'
mag_ko_path = DATA_DIR / 'mag_ko_profiles.tsv'

summary = {
    'Metric': [
        'Genomes', 'Genomes with KO annotations', 'Unique KOs', 
        'Genome-KO pairs', 'Mean KOs per genome', 'Median KOs per genome'
    ],
    'Cultured': [
        len(cult_meta),
        cult_ko['genome_id'].nunique(),
        cult_ko['ko_id'].nunique(),
        len(cult_ko),
        cult_ko.groupby('genome_id')['ko_id'].nunique().mean(),
        cult_ko.groupby('genome_id')['ko_id'].nunique().median(),
    ]
}

if mag_ko_path.exists():
    mag_meta = pd.read_csv(mag_meta_path, sep='\t')
    mag_ko = pd.read_csv(mag_ko_path, sep='\t')
    summary['MAG'] = [
        len(mag_meta),
        mag_ko['genome_id'].nunique(),
        mag_ko['ko_id'].nunique(),
        len(mag_ko),
        mag_ko.groupby('genome_id')['ko_id'].nunique().mean(),
        mag_ko.groupby('genome_id')['ko_id'].nunique().median(),
    ]

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(DATA_DIR / 'cohort_summary.tsv', sep='\t', index=False)

## 2. Headline Numbers

In [ ]:
coverage_path = DATA_DIR / 'ko_cultivation_coverage_full.tsv'

if coverage_path.exists():
    cov = pd.read_csv(coverage_path, sep='\t', index_col=0)
    sig = cov[cov['significant']]
    
    print('=== HEADLINE NUMBERS ===')
    print(f'Total KOs tested: {len(cov)}')
    print(f'Significant (FDR<0.05): {len(sig)} ({len(sig)/len(cov)*100:.1f}%)')
    print(f'  Enriched in cultured: {len(sig[sig["log2_ratio"] > 0])}')
    print(f'  Depleted in cultured: {len(sig[sig["log2_ratio"] < 0])}')
    print(f'  Cultured-only KOs (absent in MAGs): {(cov["n_mag"] == 0).sum()}')
    print(f'  MAG-only KOs (absent in cultured): {(cov["n_cult"] == 0).sum()}')
    
    marker_cov = pd.read_csv(DATA_DIR / 'marker_cultivation_coverage.tsv', sep='\t')
    print(f'\n=== HYPOTHESIS TESTING ===')
    
    # H1: Depletion of specific functions
    h1_cats = ['Wood-Ljungdahl', 'Methanogenesis', 'NiFe-hydrogenase', 'Conjugation/MGE']
    h1_data = marker_cov[marker_cov['category'].isin(h1_cats)]
    h1_depleted = h1_data[h1_data['log2_ratio'] < 0]
    print(f'H1 (depletion): {len(h1_depleted)}/{len(h1_data)} predicted KOs show depletion')
    
    # H2: Enrichment of cultivation-favorable functions
    h2_cats = ['Aerobic respiration', 'Motility/chemotaxis', 'Sulfate reduction']
    h2_data = marker_cov[marker_cov['category'].isin(h2_cats)]
    h2_enriched = h2_data[h2_data['log2_ratio'] > 0]
    print(f'H2 (enrichment): {len(h2_enriched)}/{len(h2_data)} predicted KOs show enrichment')
else:
    print('Waiting for NB04 results (need MAG annotations first)')

## 3. Combined Figure

In [ ]:
if coverage_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Panel A: Distribution of log2 ratios
    ax = axes[0]
    ax.hist(cov['log2_ratio'], bins=80, color='gray', edgecolor='none', alpha=0.7)
    ax.axvline(0, ls='-', c='black', lw=1)
    ax.set_xlabel('log2(cultured / MAG prevalence)')
    ax.set_ylabel('Number of KOs')
    ax.set_title('A. Distribution of Cultivation Bias')
    
    # Panel B: Marker category bar chart
    ax = axes[1]
    cat_means = marker_cov.groupby('category')['log2_ratio'].mean().sort_values()
    colors = ['#1f77b4' if v < 0 else '#d62728' for v in cat_means.values]
    ax.barh(range(len(cat_means)), cat_means.values, color=colors)
    ax.set_yticks(range(len(cat_means)))
    ax.set_yticklabels(cat_means.index, fontsize=8)
    ax.axvline(0, ls='-', c='black', lw=1)
    ax.set_xlabel('Mean log2 ratio')
    ax.set_title('B. Marker Categories')
    
    # Panel C: Cultured vs MAG prevalence scatter
    ax = axes[2]
    ax.scatter(cov['frac_mag'], cov['frac_cult'], s=2, alpha=0.3, c='gray')
    sig_kos = cov[cov['significant']]
    ax.scatter(sig_kos['frac_mag'], sig_kos['frac_cult'], s=3, alpha=0.5, c='red')
    ax.plot([0, 1], [0, 1], 'k--', lw=0.5)
    ax.set_xlabel('MAG prevalence')
    ax.set_ylabel('Cultured prevalence')
    ax.set_title('C. Prevalence Comparison')
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'synthesis_panel.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('Waiting for NB04 results')